In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
INPUT_ARTS = BASE_PATH + "processed_v2/articles_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("1. Khoi tao Spark Session cho Categorical CBF...")
spark = SparkSession.builder \
    .appName("Retrieval_Categorical_Profile") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

Mounted at /content/drive
1. Khoi tao Spark Session cho Categorical CBF...


In [ ]:
print("2. Doc du lieu va chia khung thoi gian...")
transactions = spark.read.parquet(INPUT_TRANS)
articles = spark.read.parquet(INPUT_ARTS)

max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

# Cua so 90 ngay de hoach dinh ro "Gu" an mac (Categorical Profile) cua khach
train_hist_start = val_start - datetime.timedelta(days=90)
test_hist_start = test_start - datetime.timedelta(days=90)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

print(f"Train History: {train_hist_start} -> {val_start}")
print(f"Test History:  {test_hist_start} -> {test_start}")

2. Doc du lieu va chia khung thoi gian...
Train History: 2020-06-10 -> 2020-09-08
Test History:  2020-06-17 -> 2020-09-15


In [ ]:
def generate_categorical_candidates(history_df, articles_df, top_n=40): # Đổi tham số mặc định lên 40
    # Buoc 1: Chon 3 dac trung phan loai co san trong file processed cua ban
    arts_meta = articles_df.select(
        "article_id",
        "gender",               # Gioi tinh
        "product_group_name",   # Nhom san pham
        "colour_group_name"     # Mau sac
    )

    # Tao ma to hop (Combo ID): VD: "Ladies_Garment Upper body_Black"
    arts_meta = arts_meta.withColumn(
        "combo_id",
        F.concat_ws("_", F.col("gender"), F.col("product_group_name"), F.col("colour_group_name"))
    )

    hist_arts = history_df.join(arts_meta, "article_id", "inner")

    # Buoc 2: Ho so Khach hang (User Profile)
    user_profile = hist_arts.groupBy("customer_id", "combo_id").agg(F.count("*").alias("user_affinity"))
    window_user_combo = Window.partitionBy("customer_id").orderBy(F.col("user_affinity").desc())

    # [TỐI ƯU 1] Tăng lên Top 5 tổ hợp yêu thích nhất để bắt được nhiều "Gu" hơn
    user_top_combos = user_profile.withColumn("rn", F.row_number().over(window_user_combo)) \
        .filter(F.col("rn") <= 5).drop("rn")

    # Buoc 3: Do nong cua San pham (Item Trending)
    hist_max_date = history_df.select(F.max("t_dat_date")).collect()[0][0]
    recent_week_start = hist_max_date - datetime.timedelta(days=14)
    recent_hist_arts = hist_arts.filter(F.col("t_dat_date") >= recent_week_start)

    combo_popularity = recent_hist_arts.groupBy("combo_id", "article_id").agg(F.count("*").alias("item_hotness"))
    window_combo_item = Window.partitionBy("combo_id").orderBy(F.col("item_hotness").desc())

    # [TỐI ƯU 2] Tăng lên Top 20 món đồ hot nhất mỗi tổ hợp (vì xô dữ liệu hiện tại khá to)
    trending_items_per_combo = combo_popularity.withColumn("rn", F.row_number().over(window_combo_item)) \
        .filter(F.col("rn") <= 20).drop("rn")

    # Buoc 4: Ghep Ho so Khach hang voi San pham
    candidates = user_top_combos.join(trending_items_per_combo, "combo_id", "inner")

    # Buoc 5: Tinh diem proxy cho Logistic Regression
    candidates = candidates.withColumn("lr_proxy_score", F.col("user_affinity") * F.col("item_hotness"))

    window_final = Window.partitionBy("customer_id").orderBy(F.col("lr_proxy_score").desc())

    # [TỐI ƯU 3] Cắt lấy Top N (hiện tại là 40) ứng viên xuất sắc nhất
    final_cands = candidates.withColumn("rn", F.row_number().over(window_final)) \
        .filter(F.col("rn") <= top_n) \
        .select("customer_id", "article_id") \
        .withColumn("strategy", F.lit("categorical_profile"))

    return final_cands

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"   -> Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
print("\n3. Generating Categorical candidates for Train set...")
train_cands = generate_categorical_candidates(train_hist_df, articles, top_n=40)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_categorical.parquet")

print("4. Generating Categorical candidates for Test set...")
test_cands = generate_categorical_candidates(test_hist_df, articles, top_n=40)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_categorical.parquet")

print("5. Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)
print("Hoan tat! Luong dac trung Categorical da duoc mo rong.")


3. Generating Categorical candidates for Train set...
4. Generating Categorical candidates for Test set...
5. Evaluating TEST set:
   -> Actuals: 207996 | Hits: 5953 | Recall: 0.0286
Hoan tat! Luong dac trung Categorical da duoc mo rong.
